# 14-phase15-sparsity-prune

**neuron Phase 15** — paradigm 의 **정적 → 동적 위상 진입** 의 첫 단계. Phase 14 까지는 dense topology + learned magnitude 였고, Phase 15 는 학습 중 **edge 자체를 영구 제거** (sparsity-driven prune).

핵심 가설:
1. **prune 후에도 학습 지속** — pruned edge 의 gradient 가 차단됨 (resurrection 방지) 으로 학습 자체가 깨지지 않음?
2. **sparsity vs final_loss trade-off** — 30% / 50% / 70% sparsity 에서 loss 가 얼마나 악화?
3. **soft degradation** — sparsity 가 증가해도 loss 가 catastrophic 하게 폭발하지 않고 gradual?
4. **post-prune recovery** — prune 직후의 loss spike 가 이후 학습으로 어느 정도 복구?

설계: full graph block (`hybrid_around_one_around_one`, Phase 14 최저 loss 구조) 위에서 **prune fraction 4 단계** × 2 seed = 8 run.
- mode: `dense` (no prune) / `prune_0.3` / `prune_0.5` / `prune_0.7`
- prune 시점: max_steps / 2 = 750 (학습 중간)
- 그 외 hyperparameter 는 Phase 14 와 동일 (공정 비교)

데이터: TinyShakespeare (char-LM, block_size=64)
시드: [42, 123]
작성일: 2026-05-27
연관: Issue [#71](https://github.com/EinSofINTEREST/GraphLM/issues/71) / Phase 14 baseline PR [#70](https://github.com/EinSofINTEREST/GraphLM/pull/70)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridGraphTransformerLM,
    HybridTransformerTrainConfig,
    count_parameters,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config + 데이터

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# Phase 14 와 동일 hyperparameter — full graph 위에서 prune 검증
HIDDEN_DIM = 128
N_HEADS = 4
FFN_DIM = 256
N_LAYERS = 4
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500
PRUNE_AT_STEP = MAX_STEPS // 2  # 750 = 학습 중간
SEEDS = [42, 123]
ARCH = "hybrid_around_one_around_one"  # Phase 14 최저 loss 구조 고정
PRUNE_FRACTIONS = [0.0, 0.3, 0.5, 0.7]  # dense, 30%, 50%, 70%

# 모델 파라미터 수 (prune 전)
m = HybridGraphTransformerLM(
    vocab_size=vocab_size,
    hidden_dim=HIDDEN_DIM,
    n_heads=N_HEADS,
    ffn_dim=FFN_DIM,
    n_layers=N_LAYERS,
    max_seq_len=BLOCK_SIZE,
    arch=ARCH,
    group_size=GROUP_SIZE,
    use_full_graph=True,
)
print(f"\nfull graph 모델 (prune 전): params = {count_parameters(m):,}")
print(f"prune 시점: step {PRUNE_AT_STEP} / {MAX_STEPS} (학습 중간)")

## 2. Sweep 실행 (4 prune fraction × 2 seed = 8 run)

In [ ]:
results = {}
for frac in PRUNE_FRACTIONS:
    for seed in SEEDS:
        key = (frac, seed)
        mode = "dense" if frac == 0.0 else f"prune_{frac}"
        print(f"\n== mode={mode} seed={seed} ==")
        cfg = HybridTransformerTrainConfig(
            dataset=dataset,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            arch=ARCH,
            use_full_graph=True,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            prune_at_step=PRUNE_AT_STEP if frac > 0 else None,
            prune_fraction=frac,
            seed=seed,
            device=device,
        )
        out = train_hybrid_transformer_lm(cfg)
        results[key] = out
        sparsity = out["final_sparsity"]
        print(
            f"  final_loss = {out['final_loss']:.4f}  (perplexity = {safe_perplexity(out['final_loss']):.2f})"
            f"  sparsity = {sparsity:.3f}"
        )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(f"{'mode':>10s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s} {'sparsity':>10s}")
print("-" * 60)
for (frac, seed), out in results.items():
    fl = out["final_loss"]
    mode = "dense" if frac == 0.0 else f"prune_{frac}"
    print(
        f"{mode:>10s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f} {out['final_sparsity']:>10.3f}"
    )

# fraction 별 평균
print("\n== Fraction summary (mean ± σ across seeds) ==")
summary = {}
for frac in PRUNE_FRACTIONS:
    vals = [results[(frac, s)]["final_loss"] for s in SEEDS]
    sparsities = [results[(frac, s)]["final_sparsity"] for s in SEEDS]
    mean = statistics.mean(vals)
    std = statistics.stdev(vals) if len(vals) > 1 else 0.0
    sp_mean = statistics.mean(sparsities)
    summary[frac] = (mean, std, sp_mean)
    mode = "dense" if frac == 0.0 else f"prune_{frac}"
    print(
        f"  {mode:>10s} {mean:.4f} ± {std:.4f}  (perplexity ≈ {safe_perplexity(mean):.2f})  sparsity={sp_mean:.3f}"
    )

# 자동 verdict
print("\n== Verdict ==")
dense_loss = summary[0.0][0]

# 1. prune 후에도 학습 지속 — 모든 prune mode 가 finite loss 로 종료
all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_1 = "PASS" if all_finite else "FAIL"
print(f"1. all-finite after prune: {all_finite}  [{verdict_1}]")

# 2. soft degradation — 70% prune 도 dense + 1.0 이내
p70_loss = summary[0.7][0]
diff_70 = p70_loss - dense_loss
verdict_2 = "PASS" if diff_70 < 1.0 else "FAIL"
print(f"2. soft degradation (70% prune ≤ dense + 1.0): diff = {diff_70:+.4f}  [{verdict_2}]")

# 3. monotonic degradation — sparsity 증가에 따라 loss 비감소
losses_by_frac = [summary[f][0] for f in PRUNE_FRACTIONS]
monotone = all(
    losses_by_frac[i] <= losses_by_frac[i + 1] + 0.05 for i in range(len(losses_by_frac) - 1)
)
verdict_3 = "PASS" if monotone else "FAIL"
print(f"3. monotonic loss vs sparsity (±0.05 tolerance): {monotone}  [{verdict_3}]")

# 4. moderate prune (30%) 은 거의 무손실 — dense + 0.1 이내
p30_loss = summary[0.3][0]
diff_30 = p30_loss - dense_loss
verdict_4 = "PASS" if diff_30 < 0.1 else "FAIL"
print(f"4. moderate prune (30%) ≤ dense + 0.1: diff = {diff_30:+.4f}  [{verdict_4}]")

# prune 발생 정보
print("\n== Prune events ==")
for (frac, seed), out in results.items():
    if out["prune_event"] is not None:
        ev = out["prune_event"]
        print(
            f"  frac={frac} seed={seed}: step={ev['step']}, total_pruned={ev['total_pruned']}, sparsity_after={ev['sparsity_after']:.3f}"
        )

## 4. Loss curve 시각화 — prune step 표시

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = {
    0.0: "tab:gray",
    0.3: "tab:blue",
    0.5: "tab:orange",
    0.7: "tab:red",
}
window = 50

for frac in PRUNE_FRACTIONS:
    losses_per_seed = [results[(frac, s)]["losses"] for s in SEEDS]
    # rolling mean — slice 시작점 +1 시프트로 window 와 divisor 일치
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    std = arr.std(dim=0)
    steps = list(range(len(mean)))
    mode = "dense" if frac == 0.0 else f"prune {int(frac * 100)}%"
    ax.plot(steps, mean, label=mode, color=colors[frac], linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=colors[frac], alpha=0.15)

# prune step 수직선
ax.axvline(
    PRUNE_AT_STEP, color="black", linestyle=":", alpha=0.5, label=f"prune @ step {PRUNE_AT_STEP}"
)

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title(
    f"Phase 15 — sparsity-driven prune ({ARCH}, full graph): dense vs 30/50/70% prune (mean ± σ over 2 seeds)"
)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase15")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. Sparsity vs final_loss trade-off

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
fracs = [f for f in PRUNE_FRACTIONS]
means = [summary[f][0] for f in fracs]
stds = [summary[f][1] for f in fracs]
sparsities = [summary[f][2] for f in fracs]

ax.errorbar(sparsities, means, yerr=stds, marker="o", capsize=4, linewidth=1.5, color="tab:blue")
for sp, m, frac in zip(sparsities, means, fracs, strict=True):
    label = "dense" if frac == 0.0 else f"{int(frac * 100)}%"
    ax.annotate(label, (sp, m), xytext=(8, -8), textcoords="offset points", fontsize=10)

ax.set_xlabel("effective sparsity (mask=0 비율)")
ax.set_ylabel("final loss (last 100 mean ± σ)")
ax.set_title("Phase 15 — sparsity vs final_loss trade-off")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(out_dir / "sparsity_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'sparsity_tradeoff.png'}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

- soft degradation 곡선의 형태 (linear / convex / cliff)?
- 30% prune 의 무손실 / 70% 의 cliff 여부?
- prune 직후 loss spike 와 회복 양상?

**Phase 16 후보**:
- **Net2Net / LiGO 식 grow** — pruned slot 에 신규 edge 추가 (function preservation 유지), dynamic grow + shrink 동시
- **RigL / SET 식 dynamic sparse training** — 매 prune step 후 같은 수의 edge 를 다른 위치에서 재할당 (constant sparsity 유지하며 topology 만 변화)
- **layer-wise 차등 prune** — attention vs FFN 별로 다른 sparsity target